In [ ]:
import sys
import copy
import torch
import numpy as np
import tensorflow as tf
import scipy as sp
import pandas as pd
import dill as pickle
import matplotlib.pyplot as plt

In [ ]:
sys.path.append("/home/zanardi/Codes/ML/ROMAr/romar/")
from romar import env
from romar import const
from romar import utils
from romar import backend as bkd
from romar.systems import BoxAd

import romar.postproc.plotting as pltt

In [ ]:
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet/")
from pycomet.networks.chemkin.plasma_cr.meno.crmodel import CRModel

In [ ]:
env_opts = {
  "backend": "numpy",
  "device": "cpu",
  "device_idx": 0,
  "nb_threads": 2,
  "epsilon": 1e-16,
  "floatx": "float64"
}
env.set(**env_opts)

In [ ]:
path_to_dtb = "/home/zanardi/Codes/ML/ROMAr/romar/examples/database"

In [ ]:
system = BoxAd(
  species={k: path_to_dtb + "/species/" + k + ".json" for k in ("Ar", "Arp", "em")},
  kin_dtb=path_to_dtb + "/rates/kin_fit.p",
  rad_dtb=path_to_dtb + "/rates/rad_fit.p",
  use_rad=True
)

In [ ]:
rho = 1e-2
T = 3.5e4
Te = 3e2
quad_mu = {
  "x": np.array([rho, T, Te]).reshape(1,-1),
  "w": np.ones((1,1))
}
tlim = [1e-12, 1e-2]

In [ ]:
t = np.geomspace(*tlim, 200)
y0, rho = system.equil.get_init_sol(quad_mu["x"].squeeze())
y0

In [ ]:
yfom, _ = system.solve_fom(t, y0, rho)

In [ ]:
w_fom = copy.deepcopy(yfom[:system.mix.nb_comp])
n_fom = system.mix.get_n(bkd.to_torch(w_fom)).numpy()
T_fom = copy.deepcopy(yfom[system.mix.nb_comp:])
T_fom[-1] = system.mix.get_Te(pe=T_fom[-1], ne=n_fom[0])

In [ ]:
for i in range(2):
  plt.semilogx(t, T_fom[i])
plt.show()
plt.close()

In [ ]:
for s in system.mix.species.values():
  print(s.name)
  for i in s.indices:
    plt.loglog(t, n_fom[i])
  plt.show()
  plt.close()

In [ ]:
species = ("em", "Ar", "Arp")
model = CRModel(
  species={k: path_to_dtb + "/species/" + k + ".json" for k in species},
  processes="./rates2.p",
  use_corr=False,
  scale_corr=False,
  species_order=species
)

Rates

In [ ]:
def get_rates_tf(Th, Te):
  Th = Th.reshape(1,1)
  Te = Te.reshape(1,1)
  return model.compute_rates(Th, Te, f=None)

def get_rates_np(Th, Te):
  Th = bkd.to_torch(Th)
  Te = bkd.to_torch(Te)
  system.mix.update_species_thermo(Th, Te)
  system.kin.update(Th, Te)
  system.rad.update(Th, Te)
  return {**system.sources.kin.rates, **system.sources.rad.rates}

In [ ]:
i = 100
Th, Te = T_fom
Thi, Tei = Th[i], Te[i]
rates_tf = get_rates_tf(Thi, Tei)
rates_np = get_rates_np(Thi, Tei)

In [ ]:
rtf = rates_tf["fwd"]["Ie"].numpy().squeeze()
rnp = rates_np["Ie"]["fwd"].numpy()[1:]
rnp = rnp[rnp>0.0]
rnp[:10], rtf[:10], (np.abs(rnp - rtf) / (rnp+1e-80)).max()

In [ ]:
def get_rhs_tf(x):
  x = tf.convert_to_tensor(x)
  n_em, n_ar1, xlin, Th, Te = tf.split(x, [1,1,32,1,1], axis=-1)
  rates = model.compute_rates(Th, Te, f=None)
  omega = model.compose_sources_mat(rates, n_em, n_ar1)
  A = model.compose_lin_ops(omega)
  xlin = tf.expand_dims(xlin, axis=-1)
  return tf.squeeze(tf.matmul(A, xlin)).numpy()

def get_rhs_np(x):
  dx = []
  for xi in x:
    n, Th, Te = xi[:-2], xi[-2], xi[-1]
    w = system.mix.get_w(n).numpy()
    pe = system.mix.get_pe(Te, n[0])
    y = np.concatenate([w, Th.reshape(1), pe.reshape(1)])
    dxi = system.fun(0.0, y)
    dxi = system.mix.get_n(dxi[:-2])
    dx.append(dxi[2:])
  return np.vstack(dx)

In [ ]:
x = np.vstack([n_fom, T_fom]).T
dx_tf = get_rhs_tf(x)
dx_np = get_rhs_np(x)

In [ ]:
dx_np, dx_tf